# 03 · Models and Model Routing

> **The story:** Some of Sara's tasks are trivial (is this email a rent question?), some need real reasoning. Paying for the strongest model every time is wasteful.

**You will learn**
- Model Router picks a model for each request
- How to compare it with a fixed model on quality, speed and tokens
- Why you test routing per workload instead of adopting it everywhere

**Time:** about 10 minutes · **Before you start:**
- Notebook 00 finished

<sub>Workshop material: the rental documents summarise Dubai rules for training only. Not legal advice.</sub>

In [ ]:
# @title Setup: run this first (about 1 minute, then sign in)
from pathlib import Path
import os, shutil, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # The workshop code goes on this notebook's temporary Colab machine (a few seconds; nothing in Drive).
    ROOT = Path("/content/dld-foundry-workshop")
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=False)   # pick up any fixes
    else:
        subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/NetwaysPresales/dld-foundry-workshop", str(ROOT)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)])
else:
    here = Path.cwd().resolve()
    ROOT = next((p for p in [here, *here.parents] if (p / "dld_workshop").exists()), here)
sys.path.insert(0, str(ROOT))

from azure.identity import AzureCliCredential, DeviceCodeCredential
from dld_workshop import WorkshopEnvironment, WorkshopSettings
from dld_workshop.display import show, table, checklist

# In Colab you'll see a link and a code: open the link, enter the code, sign in with your work account.
credential = DeviceCodeCredential() if IN_COLAB else AzureCliCredential()

settings = WorkshopEnvironment.discover(credential)
print(f"✅ Signed in. Your environment: {settings.resource_group}")

## Step 1 · Three tasks, two deployments

An easy, a medium and a hard task each go to the fixed `gpt-5.4-mini` and to `model-router`. The **serving model** column shows which model actually answered.

In [ ]:
from dld_workshop.model_routing import compare_models, DEFAULT_ROUTING_PROMPTS

rows = compare_models(settings, credential, DEFAULT_ROUTING_PROMPTS)
table(rows, ["complexity", "deployment", "serving_model", "latency_seconds", "input_tokens", "output_tokens"])

In [ ]:
for row in rows:
    text = row["answer"] or ""
    show(f"{row['complexity']} · {row['deployment']} → {row['serving_model']}",
         text[:1500] + (" …" if len(text) > 1500 else ""))

## Step 2 · Decide with evidence
Model Router isn't automatically better. Keep it where measured quality, cost and latency beat a fixed model on **your** test set.

In this workshop the **Rules agent** uses Model Router, while the **manager**, **Mail** and **Calendar** agents use a fixed `gpt-5.4-mini`: they make tool calls in strict sequences, and in testing a fixed model was more predictable for that.

> ### 👉 Follow along in the portal
>
> **Models → `model-router`**: see the routing mode (*balanced*) and the model subset it may choose from. **Models → Compare** lets you try the same prompt side by side.

---
## What to notice
- Easy tasks are typically served by a small model; the reasoning task may go to a stronger one.
- Token counts and latency differ, which is where the savings (or costs) come from.

**Try it yourself:** add your own task to `DEFAULT_ROUTING_PROMPTS` and re-run.